# Fake News Project

## Part 1 – Data Processing

This notebook performs the following steps:

1. Load and structure the dataset
2. Clean and preprocess the text data
3. Apply the preprocessing pipeline to the 995K FakeNewsCorpus subset
4. Create train / validation / test splits (80 / 10 / 10)

### Imports and setup

In [4]:
import re
import pandas as pd
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)
import numpy as np
import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split

# Download NLTK resources if needed
nltk.download("punkt")
nltk.download("stopwords")

STOP_WORDS = set(stopwords.words("english"))
STEMMER = PorterStemmer()

INFO: Pandarallel will run on 8 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/celinaholbaekhansen/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/celinaholbaekhansen/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Helper functions

In [ ]:
def clean_content_df(df, text_col="content"):
    """Keep rows with non-missing, non-empty text and ensure text is string."""
    df2 = df.dropna(subset=[text_col]).copy()
    df2[text_col] = df2[text_col].astype(str)
    df2 = df2[df2[text_col].str.strip().ne("")].copy()
    return df2

def tokenize_series(text_series):
    """Tokenize each document into lowercase tokens and return one flat list."""
    all_tokens = []
    for text in text_series:
        all_tokens.extend(word_tokenize(text.lower()))
    return all_tokens

def remove_stopwords(tokens):
    return [token for token in tokens if token not in STOP_WORDS]

def stem_tokens(tokens):
    return [STEMMER.stem(token) for token in tokens]

def vocab_size(tokens):
    return len(set(tokens))

def reduction_rate(before, after):
    return (before - after) / before if before else 0

def count_urls(text):
    return len(re.findall(r"https?://\S+|www\.\S+", str(text)))

def count_dates(text):
    patterns = [
        r"\b\d{4}-\d{2}-\d{2}\b",              # 2024-02-17
        r"\b\d{1,2}/\d{1,2}/\d{2,4}\b",        # 17/02/2024
        r"\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s+\d{1,2},?\s+\d{4}\b"
    ]
    return sum(len(re.findall(p, str(text), flags=re.IGNORECASE)) for p in patterns)

def count_numbers(text):
    return len(re.findall(r"\b\d+(?:[\.,]\d+)?\b", str(text)))

def top_n_words(tokens, n=100):
    return pd.Series(tokens).value_counts().head(n)

## Task 1 – Retrieve, structure, clean, and preprocess the sample dataset

In [ ]:
sample_url = "https://raw.githubusercontent.com/several27/FakeNewsCorpus/master/news_sample.csv"
df_sample = pd.read_csv(sample_url)

print("Shape (rows, cols):", df_sample.shape)
print("Columns:", df_sample.columns.tolist())

df_sample.head(3)

In [ ]:
text_col = "content"

print("Missing content:", df_sample[text_col].isna().sum())

df_sample_clean = clean_content_df(df_sample, text_col=text_col)

print("Before cleaning:", df_sample.shape)
print("After cleaning: ", df_sample_clean.shape)

df_sample_clean[[text_col, "type", "domain"]].head(3)

In [ ]:
# Tokenization on sample dataset
tokens_sample = tokenize_series(df_sample_clean["content"])
vocab_sample_before = vocab_size(tokens_sample)

print("Vocabulary size before stopword removal:", vocab_sample_before)

In [ ]:
# Stopword removal on sample dataset
tokens_sample_no_stop = remove_stopwords(tokens_sample)
vocab_sample_after_stop = vocab_size(tokens_sample_no_stop)
reduction_sample_stop = reduction_rate(vocab_sample_before, vocab_sample_after_stop)

print("Vocabulary size after stopword removal:", vocab_sample_after_stop)
print("Reduction rate after stopword removal:", reduction_sample_stop)

In [ ]:
# Stemming on sample dataset
tokens_sample_stem = stem_tokens(tokens_sample_no_stop)
vocab_sample_after_stem = vocab_size(tokens_sample_stem)
reduction_sample_stem = reduction_rate(vocab_sample_before, vocab_sample_after_stem)

print("Vocabulary size after stemming:", vocab_sample_after_stem)
print("Reduction rate after stemming:", reduction_sample_stem)

## Task 2 – Apply the preprocessing pipeline to the 995K dataset

In [ ]:
import pandas as pd
import re
import nltk
from pandarallel import pandarallel
from cleantext import clean
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

# Initialize parallel processing
pandarallel.initialize(progress_bar=True)


# Load data
file_path = (r"/users/celinaholbaekhansen/Downloads/995,000_rows.csv")
df = pd.read_csv(file_path)

df = df.dropna(subset=["content"])
df["content"] = df["content"].astype(str)
df = df[df["content"].str.strip().ne("")].copy()

# Setup stopwords and stemmer
stop_words = set(stopwords.words('english'))
stop_words.update(['number', 'numb'])
stemmer = SnowballStemmer("english")


# CLEANING FUNCTION

def clean_text_library(text):
    if pd.isnull(text):
        return ""
    text = re.sub(r'http\S+|www\S+', '', text)
    text = clean(
        text,
        lower=True,
        no_urls=True,
        no_emails=True,
        no_numbers=True,
        no_line_breaks=True
    )
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["clean_text"] = df["content"].parallel_apply(clean_text_library)


#Remove stopwords

def remove_stopwords(text):
    return " ".join([w for w in text.split() if w not in stop_words])

df["no_stopwords"] = df["clean_text"].parallel_apply(remove_stopwords)

#stemming
def apply_stemming(text):
    return " ".join([stemmer.stem(w) for w in text.split()])

df["stemmed"] = df["no_stopwords"].parallel_apply(apply_stemming)


def build_vocab(series, batch_size=50000):
    vocab = set()
    for i in range(0, len(series), batch_size):
        batch = series.iloc[i:i+batch_size]
        for text in batch:
            vocab.update(text.split())
    return vocab

vocab_original = build_vocab(df["clean_text"])
vocab_no_stopwords = build_vocab(df["no_stopwords"])
vocab_stemmed = build_vocab(df["stemmed"])

# Compute sizes + reductions
size_original = len(vocab_original)
size_no_stopwords = len(vocab_no_stopwords)
size_stemmed = len(vocab_stemmed)

reduction_stopwords = ((size_original - size_no_stopwords) / size_original) * 100
reduction_stemming = ((size_no_stopwords - size_stemmed) / size_no_stopwords) * 100

print(f"Original Vocabulary Size: {size_original}")
print(f"Vocabulary Size after Stopwords: {size_no_stopwords}")
print(f"Reduction Rate after Stopwords: {reduction_stopwords:.2f}%")
print(f"Vocabulary Size after Stemming: {size_stemmed}")
print(f"Reduction Rate after Stemming: {reduction_stemming:.2f}%")


#df[["clean_text"]].to_csv("clean_text_lib.csv", index=False)
#df[["no_stopwords"]].to_csv("clean_text_no_stopwords.csv", index=False)
#df[["stemmed"]].to_csv("clean_text_stemmed.csv", index=False)

In [ ]:
#save processed DataFrame to feather format
#df = df.drop(columns=["Unnamed: 0"], errors="ignore")
#if "id" in df.columns:
#    df["id"] = df["id"].astype(str)

#df.reset_index(drop=True).to_feather("df_processed.feather")
#print("DataFrame saved to df_processed.feather")

## Task 3 – Explore the processed 995K dataset
Below are analyses that can support at least three non-trivial observations in the report.

In [ ]:
#load processed DataFrame from feather format
df_995k_clean = pd.read_feather("df_processed.feather")

In [ ]:
# Ensure 'type' and 'domain' are strings
df_995k_clean["type"] = df_995k_clean["type"].astype(str)
df_995k_clean["domain"] = df_995k_clean["domain"].astype(str)

print("Number of rows after cleaning:", len(df_995k_clean))
print("Unique type values:", df_995k_clean["type"].nunique())
print("Unique domains:", df_995k_clean["domain"].nunique())


In [ ]:
# Observation candidate 1: suspicious / corrupted label values in 'type'
type_counts = df_995k_clean["type"].value_counts()
print(type_counts.head(20))

timestamp_like_types = df_995k_clean[df_995k_clean["type"].str.match(r"^\d{4}-\d{2}-\d{2}", na=False)]
print("\nRows with timestamp-like values in 'type':", len(timestamp_like_types))
timestamp_like_types.head(3)

In [ ]:
# Observation candidate 2: article length variation
df_995k_clean["token_count"] = df_995k_clean["content"].apply(
    lambda x: len(re.findall(r'\b\w+\b', str(x)))
)

print(df_995k_clean["token_count"].describe())

df_995k_clean["token_count"].hist(bins=50)
plt.xlabel("Tokens per article")
plt.ylabel("Frequency")
plt.title("Distribution of article length")
plt.show()

In [ ]:
# Observation candidate 3: structural signals in text
df_995k_clean["url_count"] = df_995k_clean["content"].parallel_apply(count_urls)
df_995k_clean["date_count"] = df_995k_clean["content"].parallel_apply(count_dates)
df_995k_clean["number_count"] = df_995k_clean["content"].parallel_apply(count_numbers)

print(df_995k_clean[["url_count", "date_count", "number_count"]].describe())

In [ ]:
# Top frequent words before and after preprocessing
top100_before = top_n_words(df_995k_clean["clean_text"], n=100)
top100_after  = top_n_words(df_995k_clean["stemmed"],    n=100)

print("Top 20 words before stopword removal / stemming:")
print(top100_before.head(20).to_string())

print("\nTop 20 words after stopword removal + stemming:")
print(top100_after.head(20).to_string())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_sample = df_995k_clean["stemmed"].dropna().sample(50000, random_state=22)

tokens = df_sample.str.split().explode() # Split each document into tokens and flatten into one Series

top10000 = tokens.value_counts().head(10000)

plt.figure(figsize=(8, 5))
plt.plot(top10000.values)
plt.xlabel("Rank")
plt.ylabel("Frequency")
plt.title("Frequency of the 10,000 most frequent words")
plt.yscale("log")
plt.xscale("log")
plt.show()

## Task 4 – Create train / validation / test splits
We create 80% / 10% / 10% splits with stratification on the label column.

In [ ]:
# Remove rows without labels and optionally remove clearly corrupted labels
df_split = df_995k_clean.dropna(subset=["content", "type"]).copy()
df_split = df_split[~df_split["type"].astype(str).str.match(r"^\d{4}-\d{2}-\d{2}", na=False)].copy()

X = df_split["content"]
y = df_split["type"]

# First split: 80% train, 20% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=22,
    stratify=y
)

# Second split: split temp into 10% validation and 10% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=22,
    stratify=y_temp
)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))

In [ ]:
print("Train distribution (%):")
print((y_train.value_counts(normalize=True) * 100).round(3))

print("\nValidation distribution (%):")
print((y_val.value_counts(normalize=True) * 100).round(3))

print("\nTest distribution (%):")
print((y_test.value_counts(normalize=True) * 100).round(3))

In [ ]:
train_df = pd.DataFrame({"content": X_train, "type": y_train}).reset_index(drop=True)
val_df = pd.DataFrame({"content": X_val, "type": y_val}).reset_index(drop=True)
test_df = pd.DataFrame({"content": X_test, "type": y_test}).reset_index(drop=True)

train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv", index=False)
test_df.to_csv("test.csv", index=False)

print("Saved: train.csv, val.csv, test.csv")

In [ ]:
#Part 2
#PART 2
#binary logistic regression (fake or not fake) kun to mulige udfald
#oprette et binært datasæt med fake og reliable, mange variable der forsvinder og ikke tagets ned.
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report


#Loading data from Part 1
# Function to read large CSV in chunks
def read_data_chunked(filepath, chunksize=100000):
    chunks = []
    reader = pd.read_csv(
        filepath,
        chunksize=chunksize,
        low_memory=False,
        usecols=['content', 'type']
    )
    total_rows = 0
    for i, chunk in enumerate(reader, 1):
        total_rows += len(chunk)
        chunks.append(chunk)
        print('Appended', total_rows, 'articles')
    return pd.concat(chunks, ignore_index=True)

#label-mapping: "reliable" og "political" = 0 (real), else = 1 (fake) 
real_labels = ["reliable", "political"]
fake_labels = ["fake", "conspiracy", "rumor", "unreliable",
               "clickbait", "satire", "junksci", "hate", "bias"]
label_map = {l: 0 for l in real_labels}
label_map.update({l: 1 for l in fake_labels})

train_df = read_data_chunked("train.csv")
test_df = read_data_chunked("test.csv")

def prepare(df):
    df2 = df[df["type"].isin(label_map.keys())].copy()
    df2["label"] = df2["type"].map(label_map)
    df2["content"] = df2["content"].fillna("").str.lower() #to catch both "fake" and "Fake" + remove NaN
    return df2


train = prepare(train_df)
test  = prepare(test_df)

X_train = train["content"]
y_train = train["label"]

X_test = test["content"]
y_test = test["label"]

#Vectorization using CountVectorizer
vectoriser = CountVectorizer(max_features=10000) 
X_train_vec = vectoriser.fit_transform(X_train)
X_test_vec = vectoriser.transform(X_test)

#Train logistic regression model
model = LogisticRegression(max_iter=1000, random_state=22, solver="saga")
model.fit(X_train_vec, y_train)

#Predict on test set
y_pred = model.predict(X_test_vec)

#Evaluation
F1 = f1_score(y_test, y_pred)
print("F1-score:", F1)
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

#Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Real", "Fake"])
disp.plot()
plt.title("Confusion Matrix - FakeNews DATASET")
plt.show()

In [ ]:
#PART 3

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report
import joblib

In [ ]:
# Function to read large CSV in chunks
def read_data_chunked(filepath, chunksize=100000):
    chunks = []
    reader = pd.read_csv(
        filepath,
        chunksize=chunksize,
        low_memory=False,
        usecols=['content', 'type']
    )
    total_rows = 0
    for i, chunk in enumerate(reader, 1):
        total_rows += len(chunk)
        chunks.append(chunk)
        print('Appended', total_rows, 'articles')
    return pd.concat(chunks, ignore_index=True)


In [ ]:
# Prepare binary labelsfor the full dataset
def prepare_binary(df):
    real_labels = ["reliable", "political"]
    fake_labels = ["fake", "conspiracy", "rumor", "unreliable",
                   "clickbait", "satire", "junksci", "hate", "bias"]
    label_map = {l: 0 for l in real_labels}
    label_map.update({l: 1 for l in fake_labels})

    # Sørg for at type/content findes og er i brugbar form
    df = df.dropna(subset=["type", "content"])
    df["type"] = df["type"].astype(str).str.strip()
    df["content"] = df["content"].astype(str)

    # Behold kun labels, der findes i label_map
    df = df[df["type"].isin(label_map.keys())].copy()

    # Map labels til 0/1
    df["label"] = df["type"].map(label_map).astype(int)

    return df

In [ ]:
# Function to get features and labels from the dataset
def get_X_y(filepath):
    df = read_data_chunked(filepath)
    df = prepare_binary(df)
    return df['content'], df['label']

In [ ]:
def fit_and_save_vectorizer(X, train_path, output_vectorizer_path='tf_vectorizer.joblib'):
    vectorizer_tfidf = TfidfVectorizer(
        max_features=10000,
        norm="l2",
        sublinear_tf=True,
        min_df=5,
        max_df=0.9,
        ngram_range=(1, 2)
    )
    print('Fitting tf-idf vectorizer...')
    vectorizer_tfidf.fit(X)
    joblib.dump(vectorizer_tfidf, output_vectorizer_path)
X, y = get_X_y('train.csv')
fit_and_save_vectorizer(X, 'train.csv')

In [ ]:
def create_and_fit_svm(train_path, tfidf):
    print('Reading and preparing training data...')
    X, y = get_X_y(train_path)
    X = tfidf.transform(X)
    svm = LinearSVC(
        C=0.5,
        dual=True,
        class_weight="balanced",
        max_iter=2000,
        random_state=22
    )

    svm.fit(X, y)
    return svm

In [ ]:
def fit_vectorizer_and_svm(train_path):
    print('Reading and preparing training data...')
    X, y = get_X_y(train_path)
    vectorizer_tfidf = TfidfVectorizer(
        max_features=30000,
        norm="l2",
        sublinear_tf=True,
        smooth_idf=True,
        min_df=5,
        max_df=0.9, 
        ngram_range=(1, 2),
    )
    model_svm = LinearSVC(
        C=0.5,
        max_iter=2000,
        dual=True, 
        random_state=22
    )
    print('Fitting tf-idf vectorizer and transforming training data...')
    X = vectorizer_tfidf.fit_transform(X)
    print('Fitting SVM...')
    model_svm.fit(X, y)
    return model_svm, vectorizer_tfidf



In [ ]:
def validate_model(vectorizer_tfidf, model_svm, validation_path):
    print('Reading and preparing validation data...')
    X, y = get_X_y(validation_path)
    X = vectorizer_tfidf.transform(X)
    y_pred = model_svm.predict(X)
    f1 = f1_score(y, y_pred, average='macro')
    report = classification_report(y, y_pred)
    return f1, report


In [ ]:
# vectorizer_tfidf = joblib.load('vectorizer_tfidf.joblib')
# model_svm = create_and_fit_svm('train.csv', vectorizer_tfidf)
model_svm, vectorizer_tfidf = fit_vectorizer_and_svm('train.csv')
f1, report = validate_model(vectorizer_tfidf, model_svm, 'val.csv')
print('Macro F1-score:', f1)
print('Report:')
print(report)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Real", "Fake"])
disp.plot()
plt.title("Confusion Matrix - FakeNews DATASET")
plt.show()

In [ ]:
print(train.columns.tolist())

In [ ]:
print(test.columns.tolist())

In [ ]:
print(train.columns.tolist())
print(train.head(2))

In [ ]:
#PART 4 the advanced model on the LIAR dataset

In [ ]:

 
from sklearn.metrics import f1_score, classification_report

liar_df = pd.read_csv("test.tsv", sep="	", header=None)

liar_df.columns = ["id", "label", "statement", "subject", "speaker", 
                   "job", "state", "party", "barely_true", "false_count", "half_true", "mostly_true", "pants_fire", "context"]
liar_label_map = {
                "true": 1,
                "false": 0,
                "barely_true": 0,
                "barely-true": 0,
                "half_true": 0,
                "half-true": 0,
                "mostly_true": 1,
                "mostly-true": 1,
                "pants_fire": 0,
                "pants-fire": 0
}

liar_df["binary_label"] = liar_df["label"].map(liar_label_map)
liar_df = liar_df.dropna(subset=["binary_label"])

X_liar = liar_df["statement"].fillna("").str.lower()
y_liar = liar_df["binary_label"].astype(int)

X_liar_vec = vectoriser_tfidf.transform(X_liar)
y_pred_liar = model_svm.predict(X_liar_vec)

print("F1-score on LIAR dataset:", f1_score(y_liar, y_pred_liar))
print(classification_report(y_liar, y_pred_liar))


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
#Confusion matrix for LIAR dataset with advanced model
cm = confusion_matrix(y_liar, y_pred_liar)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Real", "Fake"])
disp.plot()
plt.title("Confusion Matrix -LIAR DATASET")
plt.show()

In [ ]:
#PART 4 with the simple model on the LIAR dataset

In [ ]:
import pandas as pd
from sklearn.metrics import f1_score, classification_report


liar_df = pd.read_csv("test.tsv", sep="	", header=None)

liar_df.columns = ["id", "label", "statement", "subject", "speaker", 
                   "job", "state", "party", "barely_true", "false_count", "half_true", "mostly_true", "pants_fire", "context"]
liar_label_map = {
                "true": 1,
                "false": 0,
                "barely_true": 0,
                "barely-true": 0,
                "half_true": 0,
                "half-true": 0,
                "mostly_true": 1,
                "mostly-true": 1,
                "pants_fire": 0,
                "pants-fire": 0
}

liar_df["binary_label"] = liar_df["label"].map(liar_label_map)
liar_df = liar_df.dropna(subset=["binary_label"])

X_liar = liar_df["statement"].fillna("").str.lower()
y_liar = liar_df["binary_label"].astype(int)

# bruger modellen fra part 2 til at evaluere på LIAR datasættet
X_liar_vec = vectoriser.transform(X_liar)
y_pred_liar = model.predict(X_liar_vec)

print("F1-score for LIAR-dataset:", f1_score(y_liar, y_pred_liar))
print(classification_report(y_liar, y_pred_liar))


In [ ]:
#Confusion matrix for LIAR dataset with simple model
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
cm = confusion_matrix(y_liar, y_pred_liar)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Real", "Fake"])
disp.plot()
plt.title("Confusion Matrix - LIAR Dataset")
plt.show()